# TCN Run21 Evaluation Notebook

This notebook is the clean `Plan A` evaluation flow for `Run21`.

Scope:
1. fixed-252 regime-stratified stochastic robustness
2. fixed-252 year-stratified stochastic robustness
3. true-horizon matched regime-stratified stochastic robustness
4. deterministic benchmarking vs equal-weight and SPY

Method rules:
- everything is fixed on one best checkpoint: `ep00426`
- no deterministic checkpoint sweep inside this notebook
- no stochastic random-start leakage
- artifacts are saved after each completed evaluation block
- Drive is the primary artifact destination under `tcn_tape_vectorized_runs/run21/evaluation/`


## 1) Setup
Bootstrap the repo, restore the saved `Run21` outputs, and import the evaluation helpers.


In [ ]:
import copy
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GIT_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
GIT_BRANCH = "feature/run17-film-drive-org-20260318"
CLONE_IF_MISSING = True
CLONE_PARENT_DIR = Path('/content')
CLONE_DIR_NAME = 'tcn_tape_vectorized_version_clean'
INSTALL_REQUIREMENTS = False
AUTO_INSTALL_MISSING_REQUIREMENTS = True

CRITICAL_RUNTIME_MODULES = {
    'pandas_ta_classic': 'pandas-ta-classic>=0.3.59',
    'fredapi': 'fredapi>=0.5.1',
    'yfinance': 'yfinance>=0.2.38',
}


def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)


def missing_runtime_requirements() -> list[str]:
    missing = []
    for module_name, requirement in CRITICAL_RUNTIME_MODULES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(requirement)
    return missing


def normalize_github_url(url: str | None) -> str | None:
    if not url:
        return url
    url = str(url).strip()
    if url.startswith('git@github.com:'):
        repo = url[len('git@github.com:'):]
        if repo.endswith('.git'):
            repo = repo[:-4]
        return f'https://github.com/{repo}.git'
    return url


def safe_cwd() -> Path | None:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        return None


def safe_resolve(path_like) -> Path | None:
    p = Path(path_like)
    try:
        if p.is_absolute():
            return p.resolve()
    except Exception:
        return None
    cwd = safe_cwd()
    if cwd is None:
        return None
    try:
        return (cwd / p).resolve()
    except Exception:
        return None


def find_repo_root() -> Path | None:
    candidate_roots = []
    seen = set()
    cwd = safe_cwd()
    if cwd is not None:
        for p in [cwd, *cwd.parents]:
            rp = safe_resolve(p)
            if rp is not None and rp not in seen:
                seen.add(rp)
                candidate_roots.append(rp)
    for p in [
        Path(globals().get('EVAL_REPO_DIR', CLONE_PARENT_DIR / CLONE_DIR_NAME)),
        CLONE_PARENT_DIR / CLONE_DIR_NAME,
        Path('/content/tcn_tape_vectorized_version_clean'),
        Path('/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean'),
    ]:
        rp = safe_resolve(p)
        if rp is not None and rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)
    for p in candidate_roots:
        if (p / 'src' / 'config.py').exists():
            return p
    return None


REPO_ROOT = find_repo_root()
GIT_REPO_URL = normalize_github_url(GIT_REPO_URL)

if REPO_ROOT is None and CLONE_IF_MISSING:
    if not GIT_REPO_URL:
        raise FileNotFoundError('Repo root not found and GIT_REPO_URL is not set.')
    CLONE_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    clone_target = CLONE_PARENT_DIR / CLONE_DIR_NAME
    if clone_target.exists() and not (clone_target / '.git').exists():
        shutil.rmtree(clone_target, ignore_errors=True)
    if not clone_target.exists():
        clone_cmd = ['git', 'clone', GIT_REPO_URL, str(clone_target)]
        print('+', ' '.join(map(str, clone_cmd)))
        clone_proc = subprocess.run(clone_cmd, text=True, capture_output=True)
        if clone_proc.stdout:
            print(clone_proc.stdout, end='')
        if clone_proc.returncode != 0:
            stderr = (clone_proc.stderr or '').strip()
            raise RuntimeError(f'git clone failed: {stderr}')
    REPO_ROOT = clone_target.resolve()

if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate repo root containing src/config.py.')

if GIT_BRANCH and (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', GIT_BRANCH], check=False)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_file = REPO_ROOT / 'requirements.txt'
missing_requirements = missing_runtime_requirements()
should_install = INSTALL_REQUIREMENTS or (AUTO_INSTALL_MISSING_REQUIREMENTS and bool(missing_requirements))
if should_install:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    if INSTALL_REQUIREMENTS and requirements_file.exists():
        run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    elif missing_requirements:
        run([sys.executable, '-m', 'pip', 'install', *missing_requirements])

from src.config import build_run21_config
from src.notebook_helpers.tcn_phase1 import (
    Phase1Dataset,
    compare_agent_vs_baseline,
    create_experiment6_result_stub,
    evaluate_experiment6_checkpoint,
    load_training_metadata_into_config,
    prepare_phase1_dataset,
)

print('REPO_ROOT:', REPO_ROOT)
print('Missing critical requirements before install:', missing_requirements)
print('Dependency install executed:', should_install)


## 2) Session Controls
Everything in this notebook is fixed on the single best checkpoint `ep00426`.


In [ ]:
RUN_ID = 'run21'
PRIMARY_EPISODE = 426
PRIMARY_CHECKPOINT_KIND = 'high_watermark'

EVAL_RANDOM_SEED = 42
EVAL_FORCE_TEST_START_DATE = '2020-01-01'
EVAL_DETERMINISTIC_MODE = 'mean'
EVAL_STOCHASTIC_MODE = 'sample'
STOCHASTIC_RANDOM_START = False

HORIZON_YEARS = [1, 2, 3, 4]
HORIZON_DAYS = {year: 252 * year for year in HORIZON_YEARS}

FIXED_252_HORIZON_DAYS = 252
CVAR_ALPHA = 0.05
FIXED_252_STOCH_RUNS = 32
MATCHED_REGIME_STOCH_RUNS = 16

REGIME_WINDOWS_PER_BUCKET = 3
YEAR_WINDOWS_PER_YEAR = 3
YEAR_WINDOW_MIN_GAP_DAYS = 42
BENCHMARK_START_OFFSETS = [0, 63, 126]

SAVE_EVAL_LOGS = True
SAVE_EVAL_ARTIFACTS = True
CAPTURE_RECENT_ARTIFACT_HOURS = 24

RUN_LIMIT_FIXED_252_REGIME = None
RUN_LIMIT_FIXED_252_YEAR = None
RUN_LIMIT_MATCHED_REGIME = None
RUN_LIMIT_BENCHMARK = None

AUTO_RESTORE_RESULTS_FROM_DRIVE = True
FORCE_RESTORE_RESULTS = False
RUN_RESULTS_ZIP_PATH = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs/run21/tcn_tape_vectorized_run21.zip')
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs/run21')
DRIVE_EVAL_PARENT = DRIVE_RUN_ROOT / 'evaluation'
EVAL_RESTORE_DIR = Path('/content/eval_restore')

print('Primary checkpoint episode:', PRIMARY_EPISODE)
print('Stochastic random-start enabled:', STOCHASTIC_RANDOM_START)
print('Fixed-252 seeds per window:', FIXED_252_STOCH_RUNS)
print('Matched-horizon regime seeds per window:', MATCHED_REGIME_STOCH_RUNS)


## 3) Restore Results and Resolve Artifact Roots
Restore the saved `Run21` zip from Drive when needed, then resolve the working results root and the Drive-backed session root.


In [ ]:
def _running_in_colab() -> bool:
    return importlib.util.find_spec('google.colab') is not None


def _maybe_mount_drive() -> None:
    if not _running_in_colab():
        return
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')


def _has_metadata(results_root: Path) -> bool:
    logs_dir = results_root / 'logs'
    return logs_dir.exists() and any(logs_dir.glob('*_metadata.json'))


def _restore_results_from_drive_if_needed() -> None:
    if not AUTO_RESTORE_RESULTS_FROM_DRIVE:
        return
    if _running_in_colab():
        _maybe_mount_drive()
    if not RUN_RESULTS_ZIP_PATH.exists():
        print('[INFO] Run results zip not found on Drive:', RUN_RESULTS_ZIP_PATH)
        return
    current_root = EVAL_RESTORE_DIR / 'tcn_fusion_results'
    if current_root.exists() and _has_metadata(current_root) and not FORCE_RESTORE_RESULTS:
        print('[OK] Existing restored results found:', current_root)
        return
    if EVAL_RESTORE_DIR.exists():
        shutil.rmtree(EVAL_RESTORE_DIR, ignore_errors=True)
    EVAL_RESTORE_DIR.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(RUN_RESULTS_ZIP_PATH, 'r') as zf:
        zf.extractall(EVAL_RESTORE_DIR)
    print('[OK] Restored results zip to:', EVAL_RESTORE_DIR)


def _find_named_dirs(base: Path, dir_name: str) -> list[Path]:
    if not base.exists():
        return []
    matches = []
    direct = base / dir_name
    if direct.exists() and direct.is_dir():
        matches.append(direct)
    try:
        for p in base.rglob(dir_name):
            if p.is_dir() and p not in matches:
                matches.append(p)
    except Exception:
        pass
    return matches


_restore_results_from_drive_if_needed()

repo_dir = Path(str(REPO_ROOT))
restore_dir = EVAL_RESTORE_DIR

results_candidates = []
for root in [restore_dir, repo_dir]:
    for candidate in _find_named_dirs(root, 'tcn_fusion_results'):
        if candidate not in results_candidates:
            results_candidates.append(candidate)

metadata_candidates = [p for p in results_candidates if _has_metadata(p)]
if metadata_candidates:
    EVAL_RESULTS_ROOT = sorted(
        metadata_candidates,
        key=lambda p: max((f.stat().st_mtime for f in (p / 'logs').glob('*_metadata.json')), default=0),
        reverse=True,
    )[0]
elif results_candidates:
    EVAL_RESULTS_ROOT = results_candidates[0]
else:
    EVAL_RESULTS_ROOT = restore_dir / 'tcn_fusion_results'

prep_candidates = []
for root in [restore_dir, EVAL_RESULTS_ROOT.parent, repo_dir]:
    for candidate in _find_named_dirs(root, 'data_exports'):
        if candidate not in prep_candidates:
            prep_candidates.append(candidate)
EVAL_PREP_ARTIFACTS_DIR = next((p for p in prep_candidates if p.exists()), repo_dir / 'data_exports')

SESSION_TAG = datetime.utcnow().strftime('run21_eval_planA_%Y%m%d_%H%M%S_utc')
if _running_in_colab() or DRIVE_EVAL_PARENT.exists() or DRIVE_EVAL_PARENT.parent.exists():
    DRIVE_EVAL_PARENT.mkdir(parents=True, exist_ok=True)
    EVAL_SESSION_ROOT = DRIVE_EVAL_PARENT / SESSION_TAG
else:
    EVAL_SESSION_ROOT = EVAL_RESULTS_ROOT / 'evaluation' / SESSION_TAG
EVAL_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

print('EVAL_RESULTS_ROOT:', EVAL_RESULTS_ROOT)
print('EVAL_PREP_ARTIFACTS_DIR:', EVAL_PREP_ARTIFACTS_DIR)
print('EVAL_SESSION_ROOT:', EVAL_SESSION_ROOT)


## 4) Build Metadata-Aligned Evaluation Config and Dataset
The evaluation config is rebuilt from `Run21` metadata before any checkpoint or window planning.


In [ ]:
eval_config = build_run21_config('phase1')


def _find_metadata_files(search_roots: list[Path]) -> list[Path]:
    files = []
    for root in search_roots:
        logs_dir = root / 'logs'
        if logs_dir.exists():
            files.extend(logs_dir.glob('*_metadata.json'))
    unique = []
    seen = set()
    for p in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(rp)
    return unique


metadata_search_roots = []
for root in [EVAL_RESULTS_ROOT, restore_dir / 'tcn_fusion_results', repo_dir / 'tcn_fusion_results']:
    if root not in metadata_search_roots:
        metadata_search_roots.append(root)

meta_files = _find_metadata_files(metadata_search_roots)
if not meta_files:
    searched = '\n'.join(str(root / 'logs') for root in metadata_search_roots)
    raise FileNotFoundError(f'No metadata JSON found. Searched:\n{searched}')

matched_meta = []
for meta_path in meta_files:
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        checkpointing = meta.get('Checkpointing', {}) or {}
        if str(checkpointing.get('high_watermark_checkpoint_subdir', '')).strip() == 'high_watermark_checkpoints_run21':
            matched_meta.append(meta_path)
    except Exception:
        pass

EVAL_METADATA_PATH = matched_meta[0] if matched_meta else meta_files[0]
EVAL_METADATA_PAYLOAD = json.loads(EVAL_METADATA_PATH.read_text(encoding='utf-8'))
print('Using metadata:', EVAL_METADATA_PATH)
load_training_metadata_into_config(EVAL_METADATA_PATH, eval_config)

run_ctx = EVAL_METADATA_PAYLOAD.get('Run_Context', {}) or {}
train_date_min = run_ctx.get('train_date_min')
test_date_min = run_ctx.get('test_date_min')
test_date_max = run_ctx.get('test_date_max')
if train_date_min:
    eval_config['ANALYSIS_START_DATE'] = str(pd.Timestamp(train_date_min).date())
if test_date_max:
    eval_config['ANALYSIS_END_DATE'] = str(pd.Timestamp(test_date_max).date())
if test_date_min:
    eval_config['TRAIN_TEST_SPLIT_DATE'] = str((pd.Timestamp(test_date_min) - pd.Timedelta(days=1)).date())
if EVAL_FORCE_TEST_START_DATE:
    forced_start = pd.Timestamp(EVAL_FORCE_TEST_START_DATE)
    eval_config['TRAIN_TEST_SPLIT_DATE'] = str((forced_start - pd.Timedelta(days=1)).date())

EVAL_HW_SUBDIR = str(eval_config.get('training_params', {}).get('high_watermark_checkpoint_subdir', 'high_watermark_checkpoints_run21'))
EVAL_STEP_SUBDIR = str(eval_config.get('training_params', {}).get('step_sharpe_checkpoint_subdir', 'step_sharpe_checkpoints_run21'))

prep_dir = Path(EVAL_PREP_ARTIFACTS_DIR)
eval_phase1_data = prepare_phase1_dataset(
    eval_config,
    force_download=False,
    save_preparation_artifacts=False,
    preparation_artifacts_dir=str(prep_dir) if prep_dir.exists() else None,
)

test_dates = pd.to_datetime(eval_phase1_data.test_df['Date']).drop_duplicates().sort_values().reset_index(drop=True)

print('Train shape:', eval_phase1_data.train_df.shape)
print('Test shape:', eval_phase1_data.test_df.shape)
print('Tickers:', eval_config['ASSET_TICKERS'])
print('Checkpoint subdirs:', EVAL_HW_SUBDIR, EVAL_STEP_SUBDIR)
print('Analysis window:', eval_config.get('ANALYSIS_START_DATE'), '->', eval_config.get('ANALYSIS_END_DATE'))
print('Train/test split date:', eval_config.get('TRAIN_TEST_SPLIT_DATE'))


## 5) Session Directories and Reusable Helpers
All stage outputs are organized under the Drive-backed session root. Tables and copied artifacts are written after each completed evaluation block.


In [ ]:
def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    return path


def load_csv_or_empty(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def make_stage_dirs(stage_name: str) -> dict:
    root = EVAL_SESSION_ROOT / stage_name
    dirs = {'root': root}
    for key in ['plans', 'tables', 'logs', 'tracks', 'weights', 'alphas', 'manifests']:
        dirs[key] = root / key
        dirs[key].mkdir(parents=True, exist_ok=True)
    return dirs


SECTION_DIRS = {
    'metadata': make_stage_dirs('00_metadata'),
    'fixed_252_regime': make_stage_dirs('01_fixed_252_regime'),
    'fixed_252_year': make_stage_dirs('02_fixed_252_year'),
    'matched_regime': make_stage_dirs('03_matched_regime'),
    'benchmarks': make_stage_dirs('04_benchmarks'),
    'cvar': make_stage_dirs('05_cvar'),
    'contained_regime': make_stage_dirs('06_contained_regime'),
    'manifest': make_stage_dirs('07_manifest'),
}

ARTIFACT_COPY_SEEN = set()


def get_eval_artifact_sources() -> list[Path]:
    candidates = [
        EVAL_RESULTS_ROOT / EVAL_HW_SUBDIR / 'logs',
        EVAL_RESULTS_ROOT / EVAL_STEP_SUBDIR / 'logs',
        EVAL_RESULTS_ROOT / 'logs',
    ]
    out = []
    for p in candidates:
        if p.exists() and p not in out:
            out.append(p)
    return out


def _artifact_target_dir(stage_dirs: dict, file_name: str) -> Path:
    name = file_name.lower()
    if '_weights_' in name:
        return stage_dirs['weights']
    if '_alphas_' in name:
        return stage_dirs['alphas']
    if '_actions_' in name or '_portfolio_' in name or '_history_' in name:
        return stage_dirs['tracks']
    return stage_dirs['logs']


def copy_eval_outputs_since(start_ts: float, stage_dirs: dict, block_prefix: str) -> list[str]:
    copied = []
    for src_dir in get_eval_artifact_sources():
        for p in sorted(src_dir.glob('*')):
            if not p.is_file():
                continue
            if p.stat().st_mtime + 1e-6 < start_ts:
                continue
            key = (str(p.resolve()), p.stat().st_mtime_ns)
            if key in ARTIFACT_COPY_SEEN:
                continue
            ARTIFACT_COPY_SEEN.add(key)
            target_dir = _artifact_target_dir(stage_dirs, p.name)
            dst = target_dir / f'{block_prefix}__{p.name}'
            shutil.copy2(p, dst)
            copied.append(str(dst.relative_to(EVAL_SESSION_ROOT)))
    return copied


def discover_checkpoint_pairs(results_root: Path, *, high_watermark_subdir: str | None = None, step_sharpe_subdir: str | None = None, include_root: bool = False) -> pd.DataFrame:
    rows = []
    search_dirs = []
    if high_watermark_subdir:
        search_dirs.append((results_root / high_watermark_subdir, 'high_watermark'))
    if step_sharpe_subdir:
        search_dirs.append((results_root / step_sharpe_subdir, 'periodic_step'))
    if include_root:
        search_dirs.append((results_root, 'root'))
    for base_dir, kind in search_dirs:
        if not base_dir.exists():
            continue
        pattern = '*_actor.weights.h5' if kind == 'root' else '**/*_actor.weights.h5'
        for actor in sorted(base_dir.glob(pattern)):
            prefix = str(actor).replace('_actor.weights.h5', '')
            critic = Path(prefix + '_critic.weights.h5')
            if not critic.exists():
                continue
            name = actor.name
            m_sh = re.search(r'_sh([pm])(\d+)p(\d+)', name)
            sharpe_tag = None
            if m_sh:
                sign = 1.0 if m_sh.group(1) == 'p' else -1.0
                sharpe_tag = sign * float(f"{m_sh.group(2)}.{m_sh.group(3)}")
            m_ep = re.search(r'_ep(\d+)', name)
            m_st = re.search(r'_step(\d+)', name)
            rows.append({
                'checkpoint_prefix': prefix,
                'actor_path': str(actor),
                'critic_path': str(critic),
                'checkpoint_kind': kind,
                'episode': int(m_ep.group(1)) if m_ep else np.nan,
                'step': int(m_st.group(1)) if m_st else np.nan,
                'sharpe_tag': sharpe_tag,
                'mtime': actor.stat().st_mtime,
            })
    if not rows:
        raise RuntimeError(f'No valid actor+critic checkpoints under {results_root}')
    return pd.DataFrame(rows).sort_values(['checkpoint_kind', 'episode', 'step', 'mtime'], ascending=[True, True, True, False]).reset_index(drop=True)


def resolve_primary_checkpoint(df_ckpt: pd.DataFrame, episode: int, checkpoint_kind: str = 'high_watermark') -> pd.Series:
    subset = df_ckpt[(df_ckpt['checkpoint_kind'] == checkpoint_kind) & (df_ckpt['episode'] == episode)].copy()
    if subset.empty:
        raise RuntimeError(f'Primary checkpoint ep{episode:04d} not found for kind={checkpoint_kind}')
    subset = subset.sort_values(['mtime', 'sharpe_tag'], ascending=[False, False]).reset_index(drop=True)
    return subset.iloc[0]


def create_primary_winners_df(primary_row: pd.Series) -> pd.DataFrame:
    label = f"{primary_row['checkpoint_kind']}__ep{int(primary_row['episode']):04d}"
    return pd.DataFrame([
        {
            'checkpoint_label': label,
            'checkpoint_prefix': primary_row['checkpoint_prefix'],
            'years': int(years),
            'horizon_days': int(horizon_days),
        }
        for years, horizon_days in HORIZON_DAYS.items()
    ])


def make_phase1_slice(base_phase1: Phase1Dataset, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df['Date'] = pd.to_datetime(test_df['Date'])
    unique_dates = pd.Series(test_df['Date'].dropna().unique()).sort_values().reset_index(drop=True)
    if start_offset >= len(unique_dates):
        return None, None
    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) < max(60, int(horizon_days * 0.5)):
        return None, None
    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])
    sliced_df = test_df[(test_df['Date'] >= d0) & (test_df['Date'] <= d1)].copy()
    if sliced_df.empty:
        return None, None
    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0
    return phase1_slice, {
        'window_start_date': str(d0.date()),
        'window_end_date': str(d1.date()),
        'n_days': int(len(win_dates)),
    }


def eval_run_one_checkpoint(eval_cfg, phase1_data, ckpt_prefix, *, seed, num_eval_runs, stochastic_limit_days, save_logs, save_artifacts, stochastic_random_start):
    stub = create_experiment6_result_stub(
        random_seed=seed,
        use_covariance=True,
        architecture=eval_cfg['agent_params']['actor_critic_type'],
        checkpoint_path=ckpt_prefix,
        agent_config=copy.deepcopy(eval_cfg['agent_params']),
        base_agent_params=None,
    )
    return evaluate_experiment6_checkpoint(
        experiment6=stub,
        phase1_data=phase1_data,
        config=eval_cfg,
        random_seed=seed,
        checkpoint_path_override=ckpt_prefix,
        deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
        num_eval_runs=int(num_eval_runs),
        stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
        stochastic_episode_length_limit=int(stochastic_limit_days),
        stochastic_random_start=bool(stochastic_random_start),
        save_eval_logs=bool(save_logs),
        save_eval_artifacts=bool(save_artifacts),
    )


REGIME_BUCKET_SPECS = [
    ('pre_covid', pd.Timestamp('1900-01-01'), pd.Timestamp('2020-03-01'), 'Pre-COVID (through 2020-02)'),
    ('covid_crash_recession', pd.Timestamp('2020-03-01'), pd.Timestamp('2020-05-01'), 'COVID Crash / NBER Recession (2020-03 to 2020-04)'),
    ('covid_recovery', pd.Timestamp('2020-05-01'), pd.Timestamp('2021-01-01'), 'COVID Recovery (2020-05 to 2020-12)'),
    ('post_pandemic_rally', pd.Timestamp('2021-01-01'), pd.Timestamp('2022-01-01'), 'Post-Pandemic Rally (2021)'),
    ('inflation_shock_fed_tightening', pd.Timestamp('2022-01-01'), pd.Timestamp('2023-07-27'), 'Inflation Shock / Fed Tightening (2022 to 2023-07-26)'),
    ('disinflation_higher_for_longer', pd.Timestamp('2023-07-27'), pd.Timestamp('2024-09-19'), 'Disinflation / Higher for Longer (2023-07-27 to 2024-09-18)'),
    ('rate_cut_ai_bull', pd.Timestamp('2024-09-19'), pd.Timestamp('2100-01-01'), 'Rate Cuts / AI Bull (2024-09-19+)'),
]


def _regime_label(ts: pd.Timestamp) -> str:
    ts = pd.Timestamp(ts)
    for bucket, start_ts, end_ts, _ in REGIME_BUCKET_SPECS:
        if start_ts <= ts < end_ts:
            return bucket
    return 'unknown'


def _regime_display_label(bucket: str) -> str:
    for slug, _, _, label in REGIME_BUCKET_SPECS:
        if slug == bucket:
            return label
    return str(bucket)


def _expected_market_regime(start_date: str) -> str:
    return _regime_display_label(_regime_label(pd.Timestamp(start_date)))


def build_regime_windows(date_series: pd.Series, horizon_days: int, windows_per_bucket: int = 3) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(date_series)).dropna().reset_index(drop=True)
    max_start = len(dates) - int(horizon_days)
    if max_start < 0:
        return pd.DataFrame(columns=['bucket', 'start_offset', 'horizon_days', 'start_date', 'end_date', 'window_id'])
    start_df = pd.DataFrame({'offset': np.arange(max_start + 1), 'Date': dates.iloc[: max_start + 1].values})
    start_df['bucket'] = start_df['Date'].map(_regime_label)
    rows = []
    for bucket, group in start_df.groupby('bucket'):
        group = group.reset_index(drop=True)
        if group.empty:
            continue
        pick_count = min(int(windows_per_bucket), len(group))
        pick_positions = np.linspace(0, len(group) - 1, num=pick_count).round().astype(int)
        chosen_offsets = []
        for pos in sorted(set(pick_positions.tolist())):
            row = group.iloc[pos]
            offset = int(row['offset'])
            if offset in chosen_offsets:
                continue
            chosen_offsets.append(offset)
            start_date = pd.Timestamp(dates.iloc[offset])
            end_date = pd.Timestamp(dates.iloc[min(len(dates) - 1, offset + int(horizon_days) - 1)])
            rows.append({
                'bucket': bucket,
                'start_offset': offset,
                'horizon_days': int(horizon_days),
                'start_date': str(start_date.date()),
                'end_date': str(end_date.date()),
            })
    out = pd.DataFrame(rows).sort_values(['bucket', 'start_offset']).reset_index(drop=True)
    if not out.empty:
        out['window_id'] = [f'regime_{idx:03d}' for idx in range(1, len(out) + 1)]
    return out


def build_year_windows(date_series: pd.Series, horizon_days: int, windows_per_year: int = 3, min_gap_days: int = 42) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(date_series)).dropna().reset_index(drop=True)
    max_start = len(dates) - int(horizon_days)
    if max_start < 0:
        return pd.DataFrame(columns=['bucket', 'start_offset', 'horizon_days', 'start_date', 'end_date', 'window_id'])
    start_df = pd.DataFrame({'offset': np.arange(max_start + 1), 'Date': dates.iloc[: max_start + 1].values})
    start_df['bucket'] = pd.to_datetime(start_df['Date']).dt.year.astype(int).astype(str)
    rows = []
    for bucket, group in start_df.groupby('bucket'):
        group = group.reset_index(drop=True)
        if group.empty:
            continue
        pick_count = min(int(windows_per_year), len(group))
        pick_positions = np.linspace(0, len(group) - 1, num=pick_count).round().astype(int)
        chosen_dates = []
        for pos in sorted(set(pick_positions.tolist())):
            row = group.iloc[pos]
            start_date = pd.Timestamp(row['Date'])
            if any(abs((start_date - prev).days) < int(min_gap_days) for prev in chosen_dates):
                continue
            chosen_dates.append(start_date)
            offset = int(row['offset'])
            end_date = pd.Timestamp(dates.iloc[min(len(dates) - 1, offset + int(horizon_days) - 1)])
            rows.append({
                'bucket': bucket,
                'start_offset': offset,
                'horizon_days': int(horizon_days),
                'start_date': str(start_date.date()),
                'end_date': str(end_date.date()),
            })
    out = pd.DataFrame(rows).sort_values(['bucket', 'start_offset']).reset_index(drop=True)
    if not out.empty:
        out['window_id'] = [f'year_{idx:03d}' for idx in range(1, len(out) + 1)]
    return out


def build_contained_regime_windows(date_series: pd.Series, horizon_days: int, windows_per_bucket: int = 3) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(date_series)).dropna().reset_index(drop=True)
    if dates.empty:
        return pd.DataFrame(columns=['bucket', 'start_offset', 'horizon_days', 'start_date', 'end_date', 'window_id'])
    rows = []
    for bucket, bucket_start, bucket_end, _label in REGIME_BUCKET_SPECS:
        bucket_start = pd.Timestamp(bucket_start)
        bucket_end = pd.Timestamp(bucket_end)
        bucket_offsets = []
        for offset, start_date in enumerate(dates):
            if not (bucket_start <= start_date < bucket_end):
                continue
            end_idx = min(len(dates) - 1, offset + int(horizon_days) - 1)
            end_date = pd.Timestamp(dates.iloc[end_idx])
            if end_date >= bucket_end:
                continue
            bucket_offsets.append((offset, start_date, end_date))
        if not bucket_offsets:
            continue
        pick_count = min(int(windows_per_bucket), len(bucket_offsets))
        pick_positions = np.linspace(0, len(bucket_offsets) - 1, num=pick_count).round().astype(int)
        chosen = []
        for pos in sorted(set(pick_positions.tolist())):
            offset, start_date, end_date = bucket_offsets[pos]
            if offset in chosen:
                continue
            chosen.append(offset)
            rows.append({
                'bucket': bucket,
                'start_offset': int(offset),
                'horizon_days': int(horizon_days),
                'start_date': str(pd.Timestamp(start_date).date()),
                'end_date': str(pd.Timestamp(end_date).date()),
            })
    out = pd.DataFrame(rows).sort_values(['bucket', 'start_offset']).reset_index(drop=True)
    if not out.empty:
        out['window_id'] = [f'contained_{idx:03d}' for idx in range(1, len(out) + 1)]
    return out


def build_matched_regime_plan(date_series: pd.Series, horizon_days_map: dict[int, int], windows_per_bucket: int = 3) -> pd.DataFrame:
    parts = []
    for years, horizon_days in horizon_days_map.items():
        win_df = build_regime_windows(date_series, horizon_days=horizon_days, windows_per_bucket=windows_per_bucket)
        if win_df.empty:
            continue
        win_df = win_df.copy()
        win_df['years'] = int(years)
        win_df['window_id'] = [f'matched_y{years}_{idx:03d}' for idx in range(1, len(win_df) + 1)]
        parts.append(win_df)
    if not parts:
        return pd.DataFrame(columns=['years', 'bucket', 'start_offset', 'horizon_days', 'start_date', 'end_date', 'window_id'])
    return pd.concat(parts, ignore_index=True).sort_values(['years', 'bucket', 'start_offset']).reset_index(drop=True)


def summarize_stochastic_windows(raw_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if raw_df.empty:
        return pd.DataFrame()
    df = raw_df.copy()
    if 'max_drawdown_abs' in df.columns and 'max_drawdown' not in df.columns:
        df['max_drawdown'] = df['max_drawdown_abs']
    agg = {}
    for col, base in [('sharpe_ratio', 'sharpe'), ('total_return', 'return'), ('max_drawdown', 'mdd'), ('turnover', 'turnover')]:
        if col in df.columns:
            agg[f'{base}_mean'] = (col, 'mean')
            agg[f'{base}_median'] = (col, 'median')
            agg[f'{base}_std'] = (col, 'std')
            agg[f'{base}_min'] = (col, 'min')
            agg[f'{base}_max'] = (col, 'max')
    if 'sharpe_ratio' in df.columns:
        df['negative_sharpe'] = (df['sharpe_ratio'] < 0).astype(int)
        agg['negative_sharpe_count'] = ('negative_sharpe', 'sum')
        agg['negative_sharpe_rate'] = ('negative_sharpe', 'mean')
    return df.groupby(group_cols, as_index=False).agg(**agg)


def summarize_bucket_degradation(window_summary_df: pd.DataFrame) -> pd.DataFrame:
    if window_summary_df.empty:
        return pd.DataFrame()
    df = window_summary_df.copy()
    if 'det_sharpe' in df.columns and 'sharpe_mean' in df.columns:
        df['sharpe_degradation'] = df['sharpe_mean'] - df['det_sharpe']
    if 'det_return' in df.columns and 'return_mean' in df.columns:
        df['return_degradation'] = df['return_mean'] - df['det_return']
    group_cols = [c for c in ['checkpoint_label', 'checkpoint_prefix', 'years', 'bucket'] if c in df.columns]
    keep_cols = [c for c in ['sharpe_mean', 'sharpe_median', 'negative_sharpe_rate', 'mdd_mean', 'turnover_mean', 'sharpe_degradation', 'return_degradation'] if c in df.columns]
    if not group_cols or not keep_cols:
        return pd.DataFrame()
    agg = {col: 'mean' for col in keep_cols}
    return df.groupby(group_cols, as_index=False).agg(agg)


def compute_tail_risk_metrics(returns, alpha: float = CVAR_ALPHA) -> dict:
    series = pd.Series(returns, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    if series.empty:
        return {
            'num_obs': 0,
            'mean_return': np.nan,
            'std_return': np.nan,
            'var_alpha': np.nan,
            'cvar_alpha': np.nan,
            'var_alpha_pct': np.nan,
            'cvar_alpha_pct': np.nan,
        }
    var_alpha = float(series.quantile(float(alpha)))
    tail = series[series <= var_alpha]
    cvar_alpha = float(tail.mean()) if not tail.empty else var_alpha
    return {
        'num_obs': int(series.shape[0]),
        'mean_return': float(series.mean()),
        'std_return': float(series.std(ddof=0)),
        'var_alpha': var_alpha,
        'cvar_alpha': cvar_alpha,
        'var_alpha_pct': var_alpha * 100.0,
        'cvar_alpha_pct': cvar_alpha * 100.0,
    }


def _returns_from_portfolio_track(df: pd.DataFrame) -> pd.Series:
    if 'daily_return' in df.columns:
        return pd.to_numeric(df['daily_return'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if 'portfolio_value' in df.columns:
        pv = pd.to_numeric(df['portfolio_value'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if len(pv) >= 2:
            return pv.pct_change().dropna()
    return pd.Series(dtype=float)


def _append_deduped_csv(path: Path, new_df: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    existing = load_csv_or_empty(path)
    combined = pd.concat([existing, new_df], ignore_index=True, sort=False)
    if key_cols:
        missing = [c for c in key_cols if c not in combined.columns]
        if not missing:
            combined = combined.drop_duplicates(subset=key_cols, keep='last')
    save_df(combined, path)
    return combined


def _parse_stage_portfolio_artifact(path: Path, stage_key: str) -> dict | None:
    pattern = re.compile(
        rf'^{re.escape(stage_key)}__bucket-(?P<bucket>.+?)__window-(?P<window_id>.+?)__start-(?P<start_date>\d{{4}}-\d{{2}}-\d{{2}})__h(?P<horizon_days>\d+)__ep(?P<episode>\d+)__.*_portfolio_(?P<track_name>.+)\.csv$'
    )
    m = pattern.match(path.name)
    if not m:
        return None
    out = m.groupdict()
    out['checkpoint_episode'] = int(out.pop('episode'))
    out['horizon_days'] = int(out['horizon_days'])
    out['years'] = max(1, int(round(out['horizon_days'] / 252)))
    out['track_role'] = 'stochastic' if out['track_name'] == 'stochastic' else 'deterministic'
    out['checkpoint_label'] = PRIMARY_CHECKPOINT_LABEL
    out['checkpoint_prefix'] = PRIMARY_CHECKPOINT_PREFIX
    out['expected_market_regime'] = _regime_display_label(out['bucket'])
    out['artifact_relpath'] = str(path.relative_to(EVAL_SESSION_ROOT))
    return out


def refresh_stage_cvar_tables(stage_key: str, stage_dirs: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    for track_path in sorted(stage_dirs['tracks'].glob(f'{stage_key}__*_portfolio_*.csv')):
        meta = _parse_stage_portfolio_artifact(track_path, stage_key)
        if meta is None:
            continue
        try:
            df_track = pd.read_csv(track_path)
        except Exception:
            continue
        run_groups = [('det', df_track)]
        if 'run' in df_track.columns:
            run_groups = [(str(run_id), grp.copy()) for run_id, grp in df_track.groupby('run', sort=True)]
        for run_id, grp in run_groups:
            returns = _returns_from_portfolio_track(grp)
            stats = compute_tail_risk_metrics(returns, alpha=CVAR_ALPHA)
            start_date = pd.to_datetime(grp['date'], errors='coerce').min() if 'date' in grp.columns else pd.NaT
            end_date = pd.to_datetime(grp['date'], errors='coerce').max() if 'date' in grp.columns else pd.NaT
            rows.append({
                **meta,
                'run': run_id,
                'window_start_date': str(start_date.date()) if pd.notna(start_date) else meta['start_date'],
                'window_end_date': str(end_date.date()) if pd.notna(end_date) else meta['start_date'],
                **stats,
            })
    raw_df = pd.DataFrame(rows)
    raw_path = stage_dirs['tables'] / f'{stage_key}_cvar_raw.csv'
    if raw_df.empty:
        save_df(raw_df, raw_path)
        empty = pd.DataFrame()
        save_df(empty, stage_dirs['tables'] / f'{stage_key}_cvar_window_summary.csv')
        save_df(empty, stage_dirs['tables'] / f'{stage_key}_cvar_bucket_summary.csv')
        return raw_df, empty, empty
    raw_df = raw_df.sort_values(['window_id', 'track_role', 'run']).reset_index(drop=True)
    save_df(raw_df, raw_path)

    window_group_cols = ['checkpoint_label', 'checkpoint_prefix', 'checkpoint_episode', 'years', 'bucket', 'expected_market_regime', 'window_id', 'horizon_days', 'track_role', 'window_start_date', 'window_end_date']
    window_summary = (
        raw_df.groupby(window_group_cols, as_index=False)
        .agg(
            run_count=('run', 'count'),
            num_obs_mean=('num_obs', 'mean'),
            mean_return_mean=('mean_return', 'mean'),
            var_alpha_pct_mean=('var_alpha_pct', 'mean'),
            var_alpha_pct_median=('var_alpha_pct', 'median'),
            var_alpha_pct_min=('var_alpha_pct', 'min'),
            cvar_alpha_pct_mean=('cvar_alpha_pct', 'mean'),
            cvar_alpha_pct_median=('cvar_alpha_pct', 'median'),
            cvar_alpha_pct_min=('cvar_alpha_pct', 'min'),
        )
    )
    save_df(window_summary, stage_dirs['tables'] / f'{stage_key}_cvar_window_summary.csv')

    bucket_group_cols = ['checkpoint_label', 'checkpoint_prefix', 'checkpoint_episode', 'years', 'bucket', 'expected_market_regime', 'track_role']
    bucket_summary = (
        window_summary.groupby(bucket_group_cols, as_index=False)
        .agg(
            window_count=('window_id', 'count'),
            run_count=('run_count', 'sum'),
            var_alpha_pct_mean=('var_alpha_pct_mean', 'mean'),
            var_alpha_pct_worst=('var_alpha_pct_min', 'min'),
            cvar_alpha_pct_mean=('cvar_alpha_pct_mean', 'mean'),
            cvar_alpha_pct_worst=('cvar_alpha_pct_min', 'min'),
        )
    )
    save_df(bucket_summary, stage_dirs['tables'] / f'{stage_key}_cvar_bucket_summary.csv')
    return raw_df, window_summary, bucket_summary


def save_baseline_track(stage_dirs: dict, block_prefix: str, label: str, returns: pd.Series) -> str | None:
    series = pd.Series(returns, dtype=float).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    if series.empty:
        return None
    df = pd.DataFrame({
        'step': np.arange(len(series)),
        'daily_return': series.astype(float),
        'portfolio_value_index': (1.0 + series.astype(float)).cumprod(),
    })
    dst = stage_dirs['tracks'] / f'{block_prefix}__baseline_{label}.csv'
    save_df(df, dst)
    return str(dst.relative_to(EVAL_SESSION_ROOT))


def make_block_prefix(stage_key: str, plan_row: pd.Series) -> str:
    bucket = re.sub(r'[^a-zA-Z0-9]+', '-', str(plan_row['bucket']).strip()).strip('-').lower()
    start_date = str(plan_row.get('start_date', 'na'))
    return f'{stage_key}__bucket-{bucket}__window-{plan_row["window_id"]}__start-{start_date}__h{int(plan_row["horizon_days"])}__ep{PRIMARY_EPISODE:04d}'


def run_stochastic_plan_row(plan_row: pd.Series, *, stage_key: str, stage_dirs: dict, num_eval_runs: int) -> dict:
    start_offset = int(plan_row['start_offset'])
    horizon_days = int(plan_row['horizon_days'])
    phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
    if phase1_slice is None:
        return {'window_id': plan_row['window_id'], 'status': 'skipped', 'reason': 'invalid_slice'}

    block_prefix = make_block_prefix(stage_key, plan_row)
    seed = int(EVAL_RANDOM_SEED + horizon_days * 100 + start_offset)
    call_started_at = time.time()
    ev = eval_run_one_checkpoint(
        eval_config,
        phase1_slice,
        PRIMARY_CHECKPOINT_PREFIX,
        seed=seed,
        num_eval_runs=num_eval_runs,
        stochastic_limit_days=horizon_days,
        save_logs=SAVE_EVAL_LOGS,
        save_artifacts=SAVE_EVAL_ARTIFACTS,
        stochastic_random_start=STOCHASTIC_RANDOM_START,
    )

    det = ev.deterministic_metrics or {}
    det_row = {
        'checkpoint_label': PRIMARY_CHECKPOINT_LABEL,
        'checkpoint_prefix': PRIMARY_CHECKPOINT_PREFIX,
        'checkpoint_episode': int(PRIMARY_EPISODE),
        'window_id': plan_row['window_id'],
        'bucket': plan_row['bucket'],
        'expected_market_regime': _expected_market_regime(meta['window_start_date']),
        'start_offset': start_offset,
        'horizon_days': horizon_days,
        'years': int(plan_row['years']) if 'years' in plan_row and pd.notna(plan_row['years']) else int(round(horizon_days / 252)),
        'window_start_date': meta['window_start_date'],
        'window_end_date': meta['window_end_date'],
        'det_return': float(det.get('total_return', np.nan)),
        'det_sharpe': float(det.get('sharpe_ratio', np.nan)),
        'det_mdd': float(det.get('max_drawdown_abs', det.get('max_drawdown', np.nan))),
        'det_turnover': float(det.get('turnover', np.nan)),
    }

    sto = ev.stochastic_results if isinstance(ev.stochastic_results, pd.DataFrame) else pd.DataFrame()
    if sto.empty:
        copied = copy_eval_outputs_since(call_started_at, stage_dirs, block_prefix)
        refresh_stage_cvar_tables(stage_key, stage_dirs)
        save_json({**det_row, 'copied_artifacts': copied, 'status': 'no_stochastic_rows'}, stage_dirs['manifests'] / f'{block_prefix}.json')
        return {'window_id': plan_row['window_id'], 'status': 'no_rows'}

    sto = sto.copy()
    if 'max_drawdown_abs' in sto.columns and 'max_drawdown' not in sto.columns:
        sto['max_drawdown'] = sto['max_drawdown_abs']
    if not bool(STOCHASTIC_RANDOM_START):
        expected_start = det_row['window_start_date']
        expected_days = int(horizon_days)
        bad_start = sto['start_date'].astype(str) != str(expected_start)
        if bad_start.any():
            raise AssertionError(f'Fixed-start stochastic mismatch for {plan_row["window_id"]}: starts={sorted(sto.loc[bad_start, "start_date"].astype(str).unique())}, expected={expected_start}')
        bad_days = pd.to_numeric(sto['days_traded'], errors='coerce').fillna(-1).astype(int) != expected_days
        if bad_days.any():
            raise AssertionError(f'Horizon mismatch for {plan_row["window_id"]}: days={sorted(pd.to_numeric(sto.loc[bad_days, "days_traded"], errors="coerce").dropna().astype(int).unique())}, expected={expected_days}')
        if 'market_regime' in sto.columns:
            expected_regime = det_row['expected_market_regime']
            bad_regime = sto['market_regime'].astype(str) != str(expected_regime)
            if bad_regime.any():
                raise AssertionError(f'Regime label mismatch for {plan_row["window_id"]}: regimes={sorted(sto.loc[bad_regime, "market_regime"].astype(str).unique())}, expected={expected_regime}')
    for key, value in det_row.items():
        sto[key] = value

    raw_path = stage_dirs['tables'] / f'{stage_key}_seed_raw.csv'
    updated_raw = _append_deduped_csv(raw_path, sto, key_cols=['window_id', 'run'])

    win_summary = summarize_stochastic_windows(sto, ['checkpoint_label', 'checkpoint_prefix', 'checkpoint_episode', 'years', 'bucket', 'expected_market_regime', 'window_id', 'horizon_days', 'window_start_date', 'window_end_date'])
    if not win_summary.empty:
        for key, value in det_row.items():
            if key not in win_summary.columns:
                win_summary[key] = value
        win_summary['sharpe_degradation'] = win_summary['sharpe_mean'] - win_summary['det_sharpe']
        win_summary['return_degradation'] = win_summary['return_mean'] - win_summary['det_return']
        window_summary_path = stage_dirs['tables'] / f'{stage_key}_window_summary.csv'
        _append_deduped_csv(window_summary_path, win_summary, key_cols=['window_id'])

    copied = copy_eval_outputs_since(call_started_at, stage_dirs, block_prefix)
    cvar_raw_df, cvar_window_df, cvar_bucket_df = refresh_stage_cvar_tables(stage_key, stage_dirs)
    save_json({
        'stage_key': stage_key,
        'block_prefix': block_prefix,
        'deterministic': det_row,
        'copied_artifacts': copied,
        'num_stochastic_rows': int(len(sto)),
        'stochastic_random_start': bool(STOCHASTIC_RANDOM_START),
        'cvar_rows': int(len(cvar_raw_df)),
        'cvar_window_rows': int(len(cvar_window_df)),
        'cvar_bucket_rows': int(len(cvar_bucket_df)),
    }, stage_dirs['manifests'] / f'{block_prefix}.json')

    return {
        'window_id': plan_row['window_id'],
        'status': 'ok',
        'num_rows': int(len(sto)),
        'copied_artifacts': len(copied),
        'det_sharpe': det_row['det_sharpe'],
    }


def run_plan_rows(plan_df: pd.DataFrame, *, stage_key: str, stage_dirs: dict, num_eval_runs: int, limit_rows=None) -> pd.DataFrame:
    if plan_df.empty:
        print(f'[INFO] {stage_key}: empty plan')
        return pd.DataFrame()
    raw_path = stage_dirs['tables'] / f'{stage_key}_seed_raw.csv'
    existing_raw = load_csv_or_empty(raw_path)
    completed = set(existing_raw['window_id'].dropna().astype(str)) if not existing_raw.empty and 'window_id' in existing_raw.columns else set()
    pending = plan_df[~plan_df['window_id'].astype(str).isin(completed)].copy().reset_index(drop=True)
    if limit_rows is not None:
        pending = pending.head(int(limit_rows))
    print(f'{stage_key}: completed={len(completed)} pending_now={len(pending)}')
    results = []
    for _, row in pending.iterrows():
        print(f"[RUN] {stage_key} -> {row['window_id']} | bucket={row['bucket']} | start={row['start_date']} | h={row['horizon_days']}")
        res = run_stochastic_plan_row(row, stage_key=stage_key, stage_dirs=stage_dirs, num_eval_runs=num_eval_runs)
        results.append(res)
        print(f"      status={res.get('status')} rows={res.get('num_rows', 0)} copied={res.get('copied_artifacts', 0)}")
    return pd.DataFrame(results)


def eval_identify_return_column(df: pd.DataFrame):
    for c in ['LogReturn_1d', 'log_return_1d', 'Return_1d', 'return_1d', 'daily_return']:
        if c in df.columns:
            return c
    return None


def eval_fetch_spy_returns(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.Series:
    try:
        import yfinance as yf
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'])
            import yfinance as yf
        except Exception:
            print('[WARN] Could not install/import yfinance; SPY benchmark disabled.')
            return pd.Series(dtype=float)
    try:
        df = yf.download('SPY', start=str(start_date.date()), end=str((end_date + pd.Timedelta(days=1)).date()), auto_adjust=True, progress=False)
        if df is None or df.empty:
            return pd.Series(dtype=float)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        ret = pd.to_numeric(close, errors='coerce').dropna().pct_change().dropna().astype(float)
        ret.index = pd.to_datetime(ret.index)
        return ret
    except Exception as e:
        print(f'[WARN] SPY fetch failed: {type(e).__name__}: {e}')
        return pd.Series(dtype=float)


def build_baselines_from_phase1(phase1_data: Phase1Dataset):
    test_df = phase1_data.test_df.copy()
    if 'Date' not in test_df.columns:
        raise ValueError('test_df must contain Date column')
    ret_col = eval_identify_return_column(test_df)
    if ret_col is None:
        raise ValueError('Could not identify return column in test_df')
    if 'log' in ret_col.lower():
        test_df['_simple_ret'] = np.expm1(test_df[ret_col].astype(float))
    else:
        test_df['_simple_ret'] = test_df[ret_col].astype(float)
    eqw = test_df.groupby('Date')['_simple_ret'].mean().sort_index().astype(float)
    dt_index = pd.to_datetime(eqw.index)
    spy = eval_fetch_spy_returns(dt_index.min(), dt_index.max())
    if not spy.empty:
        spy = spy.reindex(dt_index).fillna(0.0)
    return eqw.reset_index(drop=True), spy.reset_index(drop=True) if not spy.empty else pd.Series(dtype=float)


def baseline_slice(series: pd.Series, start_offset: int, horizon_days: int) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    end = min(len(series), int(start_offset) + int(horizon_days))
    return pd.Series(series.iloc[int(start_offset):end]).reset_index(drop=True).astype(float)


def run_benchmark_row(plan_row: pd.Series, stage_key: str, stage_dirs: dict) -> dict:
    start_offset = int(plan_row['start_offset'])
    horizon_days = int(plan_row['horizon_days'])
    phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
    if phase1_slice is None:
        return {'benchmark_id': plan_row['benchmark_id'], 'status': 'skipped', 'reason': 'invalid_slice'}
    block_prefix = f'{stage_key}__{plan_row["benchmark_id"]}__start-{meta["window_start_date"]}__h{horizon_days}__ep{PRIMARY_EPISODE:04d}'
    call_started_at = time.time()
    ev = eval_run_one_checkpoint(
        eval_config,
        phase1_slice,
        PRIMARY_CHECKPOINT_PREFIX,
        seed=int(EVAL_RANDOM_SEED + 900_000 + horizon_days + start_offset),
        num_eval_runs=0,
        stochastic_limit_days=horizon_days,
        save_logs=SAVE_EVAL_LOGS,
        save_artifacts=SAVE_EVAL_ARTIFACTS,
        stochastic_random_start=STOCHASTIC_RANDOM_START,
    )
    det = ev.deterministic_metrics or {}
    agent_returns = pd.Series(np.diff(ev.deterministic_portfolio) / ev.deterministic_portfolio[:-1] if len(ev.deterministic_portfolio) > 1 else [], dtype=float)
    agent_tail = compute_tail_risk_metrics(agent_returns, alpha=CVAR_ALPHA)
    row = {
        'benchmark_id': plan_row['benchmark_id'],
        'checkpoint_label': PRIMARY_CHECKPOINT_LABEL,
        'checkpoint_prefix': PRIMARY_CHECKPOINT_PREFIX,
        'checkpoint_episode': int(PRIMARY_EPISODE),
        'years': int(plan_row['years']),
        'horizon_days': horizon_days,
        'start_offset': start_offset,
        'window_start_date': meta['window_start_date'],
        'window_end_date': meta['window_end_date'],
        'det_return': float(det.get('total_return', np.nan)),
        'det_sharpe': float(det.get('sharpe_ratio', np.nan)),
        'det_mdd': float(det.get('max_drawdown_abs', det.get('max_drawdown', np.nan))),
        'det_turnover': float(det.get('turnover', np.nan)),
        'det_var_5': float(agent_tail['var_alpha']),
        'det_cvar_5': float(agent_tail['cvar_alpha']),
        'det_var_5_pct': float(agent_tail['var_alpha_pct']),
        'det_cvar_5_pct': float(agent_tail['cvar_alpha_pct']),
    }

    eqw_slice = baseline_slice(EVAL_BASELINE_EQW, start_offset, horizon_days)
    spy_slice = baseline_slice(EVAL_BASELINE_SPY, start_offset, horizon_days)
    eqw_tail = compute_tail_risk_metrics(eqw_slice, alpha=CVAR_ALPHA) if len(eqw_slice) > 0 else {}
    spy_tail = compute_tail_risk_metrics(spy_slice, alpha=CVAR_ALPHA) if len(spy_slice) > 0 else {}
    try:
        cmp_eqw = compare_agent_vs_baseline(ev, eqw_slice)
        for k, v in cmp_eqw.items():
            row[f'eqw_{k}'] = v
        if eqw_tail:
            row['eqw_var_5'] = float(eqw_tail['var_alpha'])
            row['eqw_cvar_5'] = float(eqw_tail['cvar_alpha'])
            row['eqw_var_5_pct'] = float(eqw_tail['var_alpha_pct'])
            row['eqw_cvar_5_pct'] = float(eqw_tail['cvar_alpha_pct'])
    except Exception as e:
        row['eqw_error'] = str(e)
    try:
        if len(spy_slice) > 0:
            cmp_spy = compare_agent_vs_baseline(ev, spy_slice)
            for k, v in cmp_spy.items():
                row[f'spy_{k}'] = v
            row['spy_var_5'] = float(spy_tail['var_alpha'])
            row['spy_cvar_5'] = float(spy_tail['cvar_alpha'])
            row['spy_var_5_pct'] = float(spy_tail['var_alpha_pct'])
            row['spy_cvar_5_pct'] = float(spy_tail['cvar_alpha_pct'])
        else:
            row['spy_error'] = 'SPY baseline unavailable'
    except Exception as e:
        row['spy_error'] = str(e)

    raw_path = stage_dirs['tables'] / f'{stage_key}_raw.csv'
    _append_deduped_csv(raw_path, pd.DataFrame([row]), key_cols=['benchmark_id'])

    copied = copy_eval_outputs_since(call_started_at, stage_dirs, block_prefix)
    baseline_artifacts = []
    eqw_rel = save_baseline_track(stage_dirs, block_prefix, 'eqw', eqw_slice)
    spy_rel = save_baseline_track(stage_dirs, block_prefix, 'spy', spy_slice) if len(spy_slice) > 0 else None
    if eqw_rel:
        baseline_artifacts.append(eqw_rel)
    if spy_rel:
        baseline_artifacts.append(spy_rel)
    save_json({'benchmark_row': row, 'copied_artifacts': copied, 'baseline_artifacts': baseline_artifacts}, stage_dirs['manifests'] / f'{block_prefix}.json')
    return {'benchmark_id': plan_row['benchmark_id'], 'status': 'ok', 'copied_artifacts': len(copied) + len(baseline_artifacts)}


def run_benchmark_plan(plan_df: pd.DataFrame, *, stage_key: str, stage_dirs: dict, limit_rows=None) -> pd.DataFrame:
    raw_path = stage_dirs['tables'] / f'{stage_key}_raw.csv'
    existing = load_csv_or_empty(raw_path)
    completed = set(existing['benchmark_id'].dropna().astype(str)) if not existing.empty and 'benchmark_id' in existing.columns else set()
    pending = plan_df[~plan_df['benchmark_id'].astype(str).isin(completed)].copy().reset_index(drop=True)
    if limit_rows is not None:
        pending = pending.head(int(limit_rows))
    print(f'{stage_key}: completed={len(completed)} pending_now={len(pending)}')
    rows = []
    for _, row in pending.iterrows():
        print(f"[RUN] {stage_key} -> {row['benchmark_id']} | years={row['years']} | offset={row['start_offset']}")
        out = run_benchmark_row(row, stage_key=stage_key, stage_dirs=stage_dirs)
        rows.append(out)
        print(f"      status={out.get('status')} copied={out.get('copied_artifacts', 0)}")
    return pd.DataFrame(rows)

save_json({
    'run_id': RUN_ID,
    'primary_episode': PRIMARY_EPISODE,
    'session_root': str(EVAL_SESSION_ROOT),
    'results_root': str(EVAL_RESULTS_ROOT),
    'metadata_path': str(EVAL_METADATA_PATH),
    'prep_artifacts_dir': str(EVAL_PREP_ARTIFACTS_DIR),
    'stochastic_random_start': bool(STOCHASTIC_RANDOM_START),
    'fixed_252_stoch_runs': FIXED_252_STOCH_RUNS,
    'matched_regime_stoch_runs': MATCHED_REGIME_STOCH_RUNS,
    'cvar_alpha': float(CVAR_ALPHA),
    'horizons_days': HORIZON_DAYS,
    'section_roots': {k: str(v['root']) for k, v in SECTION_DIRS.items()},
}, SECTION_DIRS['metadata']['manifests'] / 'session_setup.json')

print('Session root:', EVAL_SESSION_ROOT)
print('Artifact source dirs:', get_eval_artifact_sources())


## 6) Resolve the Primary Checkpoint
This notebook skips deterministic checkpoint selection and fixes the primary checkpoint to `ep00426`.


In [ ]:
eval_ckpt_df = discover_checkpoint_pairs(
    EVAL_RESULTS_ROOT,
    high_watermark_subdir=EVAL_HW_SUBDIR,
    step_sharpe_subdir=EVAL_STEP_SUBDIR,
    include_root=False,
)
PRIMARY_CHECKPOINT_ROW = resolve_primary_checkpoint(eval_ckpt_df, episode=PRIMARY_EPISODE, checkpoint_kind=PRIMARY_CHECKPOINT_KIND)
PRIMARY_CHECKPOINT_PREFIX = str(PRIMARY_CHECKPOINT_ROW['checkpoint_prefix'])
PRIMARY_CHECKPOINT_LABEL = f"{PRIMARY_CHECKPOINT_ROW['checkpoint_kind']}__ep{int(PRIMARY_CHECKPOINT_ROW['episode']):04d}"

deterministic_horizon_winners = create_primary_winners_df(PRIMARY_CHECKPOINT_ROW)

display(pd.DataFrame([PRIMARY_CHECKPOINT_ROW]))
display(deterministic_horizon_winners)

save_df(pd.DataFrame([PRIMARY_CHECKPOINT_ROW]), SECTION_DIRS['metadata']['tables'] / 'primary_checkpoint.csv')
save_df(deterministic_horizon_winners, SECTION_DIRS['metadata']['tables'] / 'primary_checkpoint_horizon_map.csv')


## 7) Build Fixed-252 Stochastic Window Plans
These are the main publication stochastic plans. All windows are fixed 252-day windows and all stochastic runs use the same fixed start date inside each block.


In [ ]:
fixed_252_regime_plan_df = build_regime_windows(
    test_dates,
    horizon_days=FIXED_252_HORIZON_DAYS,
    windows_per_bucket=REGIME_WINDOWS_PER_BUCKET,
)
fixed_252_regime_plan_df['years'] = 1
fixed_252_regime_plan_df['stage'] = 'fixed_252_regime'

display(fixed_252_regime_plan_df)
save_df(fixed_252_regime_plan_df, SECTION_DIRS['fixed_252_regime']['plans'] / 'fixed_252_regime_plan.csv')


In [ ]:
fixed_252_year_plan_df = build_year_windows(
    test_dates,
    horizon_days=FIXED_252_HORIZON_DAYS,
    windows_per_year=YEAR_WINDOWS_PER_YEAR,
    min_gap_days=YEAR_WINDOW_MIN_GAP_DAYS,
)
fixed_252_year_plan_df['years'] = 1
fixed_252_year_plan_df['stage'] = 'fixed_252_year'

display(fixed_252_year_plan_df)
save_df(fixed_252_year_plan_df, SECTION_DIRS['fixed_252_year']['plans'] / 'fixed_252_year_plan.csv')


## 8) Run Fixed-252 Regime Stochastic Robustness
Run this section independently. Results and copied artifacts are appended after each completed window.


In [ ]:
fixed_252_regime_run_df = run_plan_rows(
    fixed_252_regime_plan_df,
    stage_key='fixed_252_regime',
    stage_dirs=SECTION_DIRS['fixed_252_regime'],
    num_eval_runs=FIXED_252_STOCH_RUNS,
    limit_rows=RUN_LIMIT_FIXED_252_REGIME,
)
fixed_252_regime_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_seed_raw.csv')
fixed_252_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_window_summary.csv')

display(fixed_252_regime_run_df)
display(fixed_252_regime_window_summary_df.tail(10))


In [ ]:
fixed_252_regime_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_seed_raw.csv')
fixed_252_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_window_summary.csv')
fixed_252_regime_bucket_summary_df = summarize_bucket_degradation(fixed_252_regime_window_summary_df)

save_df(fixed_252_regime_bucket_summary_df, SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_bucket_summary.csv')

display(fixed_252_regime_bucket_summary_df)


## 9) Run Fixed-252 Year Stochastic Robustness
Run this section independently. Results and copied artifacts are appended after each completed window.


In [ ]:
fixed_252_year_run_df = run_plan_rows(
    fixed_252_year_plan_df,
    stage_key='fixed_252_year',
    stage_dirs=SECTION_DIRS['fixed_252_year'],
    num_eval_runs=FIXED_252_STOCH_RUNS,
    limit_rows=RUN_LIMIT_FIXED_252_YEAR,
)
fixed_252_year_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_seed_raw.csv')
fixed_252_year_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_window_summary.csv')

display(fixed_252_year_run_df)
display(fixed_252_year_window_summary_df.tail(10))


In [ ]:
fixed_252_year_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_seed_raw.csv')
fixed_252_year_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_window_summary.csv')
fixed_252_year_bucket_summary_df = summarize_bucket_degradation(fixed_252_year_window_summary_df)

save_df(fixed_252_year_bucket_summary_df, SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_bucket_summary.csv')

display(fixed_252_year_bucket_summary_df)


## 10) Build the True-Horizon Matched Regime Plan
This is the secondary robustness study. It reuses the regime buckets, but now each block uses the actual horizon (`1y/2y/3y/4y`) instead of fixed 252 days.


In [ ]:
matched_regime_plan_df = build_matched_regime_plan(
    test_dates,
    horizon_days_map=HORIZON_DAYS,
    windows_per_bucket=REGIME_WINDOWS_PER_BUCKET,
)
matched_regime_plan_df['stage'] = 'matched_regime'

display(matched_regime_plan_df)
save_df(matched_regime_plan_df, SECTION_DIRS['matched_regime']['plans'] / 'matched_regime_plan.csv')


## 11) Run the True-Horizon Matched Regime Stochastic Robustness
Run this section after the fixed-252 analyses. Results and copied artifacts are appended after each completed window.


In [ ]:
matched_regime_run_df = run_plan_rows(
    matched_regime_plan_df,
    stage_key='matched_regime',
    stage_dirs=SECTION_DIRS['matched_regime'],
    num_eval_runs=MATCHED_REGIME_STOCH_RUNS,
    limit_rows=RUN_LIMIT_MATCHED_REGIME,
)
matched_regime_raw_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_seed_raw.csv')
matched_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_window_summary.csv')

display(matched_regime_run_df)
display(matched_regime_window_summary_df.tail(10))


In [ ]:
matched_regime_raw_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_seed_raw.csv')
matched_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_window_summary.csv')
matched_regime_bucket_summary_df = summarize_bucket_degradation(matched_regime_window_summary_df)

save_df(matched_regime_bucket_summary_df, SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_bucket_summary.csv')

display(matched_regime_bucket_summary_df)


## 12) Deterministic Benchmark Plan
Benchmark the primary checkpoint against equal-weight and SPY across the main deterministic horizons and start offsets.


In [ ]:
EVAL_BASELINE_EQW, EVAL_BASELINE_SPY = build_baselines_from_phase1(eval_phase1_data)

benchmark_plan_rows = []
for years, horizon_days in HORIZON_DAYS.items():
    for start_offset in BENCHMARK_START_OFFSETS:
        phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
        if phase1_slice is None:
            continue
        benchmark_plan_rows.append({
            'benchmark_id': f'bench_y{years}_off{start_offset:04d}',
            'years': int(years),
            'horizon_days': int(horizon_days),
            'start_offset': int(start_offset),
            'start_date': meta['window_start_date'],
            'end_date': meta['window_end_date'],
        })

benchmark_plan_df = pd.DataFrame(benchmark_plan_rows)
display(benchmark_plan_df)
save_df(benchmark_plan_df, SECTION_DIRS['benchmarks']['plans'] / 'benchmark_plan.csv')


In [ ]:
benchmark_run_df = run_benchmark_plan(
    benchmark_plan_df,
    stage_key='benchmarks',
    stage_dirs=SECTION_DIRS['benchmarks'],
    limit_rows=RUN_LIMIT_BENCHMARK,
)
benchmark_raw_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv')

display(benchmark_run_df)
display(benchmark_raw_df.tail(10))


In [ ]:
benchmark_raw_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv')
if not benchmark_raw_df.empty:
    summary_agg = {
        'det_sharpe': 'mean',
        'det_return': 'mean',
        'det_mdd': 'mean',
        'det_turnover': 'mean',
    }
    rename_map = {
        'det_sharpe': 'det_sharpe_mean',
        'det_return': 'det_return_mean',
        'det_mdd': 'det_mdd_mean',
        'det_turnover': 'det_turnover_mean',
    }
    for col in [
        'det_var_5_pct', 'det_cvar_5_pct',
        'eqw_agent_sharpe', 'eqw_baseline_sharpe', 'eqw_var_5_pct', 'eqw_cvar_5_pct',
        'spy_agent_sharpe', 'spy_baseline_sharpe', 'spy_var_5_pct', 'spy_cvar_5_pct',
    ]:
        if col in benchmark_raw_df.columns:
            summary_agg[col] = 'mean'
            rename_map[col] = f'{col}_mean'
    benchmark_summary_df = (
        benchmark_raw_df
        .groupby(['checkpoint_label', 'checkpoint_prefix', 'years'], as_index=False)
        .agg(summary_agg)
        .rename(columns=rename_map)
    )
else:
    benchmark_summary_df = pd.DataFrame()

save_df(benchmark_summary_df, SECTION_DIRS['benchmarks']['tables'] / 'benchmark_summary.csv')
display(benchmark_summary_df)


## 13) CVaR From Saved Tracks
Rebuild tail-risk tables from the saved portfolio-track artifacts. This section is post-processing only and does not rerun policy evaluation.


In [ ]:
fixed_252_regime_cvar_raw_df, fixed_252_regime_cvar_window_summary_df, fixed_252_regime_cvar_bucket_summary_df = refresh_stage_cvar_tables('fixed_252_regime', SECTION_DIRS['fixed_252_regime'])
fixed_252_year_cvar_raw_df, fixed_252_year_cvar_window_summary_df, fixed_252_year_cvar_bucket_summary_df = refresh_stage_cvar_tables('fixed_252_year', SECTION_DIRS['fixed_252_year'])
matched_regime_cvar_raw_df, matched_regime_cvar_window_summary_df, matched_regime_cvar_bucket_summary_df = refresh_stage_cvar_tables('matched_regime', SECTION_DIRS['matched_regime'])

cvar_overview_parts = []
for stage_name, bucket_df in [
    ('fixed_252_regime', fixed_252_regime_cvar_bucket_summary_df),
    ('fixed_252_year', fixed_252_year_cvar_bucket_summary_df),
    ('matched_regime', matched_regime_cvar_bucket_summary_df),
]:
    if bucket_df is None or bucket_df.empty:
        continue
    part = bucket_df.copy()
    part.insert(0, 'stage', stage_name)
    cvar_overview_parts.append(part)

cvar_stage_overview_df = pd.concat(cvar_overview_parts, ignore_index=True, sort=False) if cvar_overview_parts else pd.DataFrame()
save_df(cvar_stage_overview_df, SECTION_DIRS['cvar']['tables'] / 'stochastic_cvar_stage_overview.csv')
if not benchmark_summary_df.empty:
    benchmark_cvar_summary_df = benchmark_summary_df.copy()
else:
    benchmark_cvar_summary_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmark_summary.csv')
save_df(benchmark_cvar_summary_df, SECTION_DIRS['cvar']['tables'] / 'benchmark_cvar_summary.csv')

display(cvar_stage_overview_df.head(20))
display(benchmark_cvar_summary_df)


## 14) Session Manifest and Quick Reload Handles
Save a final manifest and reload the main in-memory tables for quick inspection.


In [ ]:
manifest = {
    'run_id': RUN_ID,
    'session_tag': SESSION_TAG,
    'session_root': str(EVAL_SESSION_ROOT),
    'results_root': str(EVAL_RESULTS_ROOT),
    'metadata_path': str(EVAL_METADATA_PATH),
    'primary_checkpoint_episode': int(PRIMARY_EPISODE),
    'primary_checkpoint_label': PRIMARY_CHECKPOINT_LABEL,
    'primary_checkpoint_prefix': PRIMARY_CHECKPOINT_PREFIX,
    'stochastic_random_start': bool(STOCHASTIC_RANDOM_START),
    'fixed_252_horizon_days': int(FIXED_252_HORIZON_DAYS),
    'fixed_252_stoch_runs': int(FIXED_252_STOCH_RUNS),
    'matched_regime_stoch_runs': int(MATCHED_REGIME_STOCH_RUNS),
    'cvar_alpha': float(CVAR_ALPHA),
    'section_roots': {k: str(v['root']) for k, v in SECTION_DIRS.items()},
}
save_json(manifest, SECTION_DIRS['manifest']['manifests'] / 'analysis_manifest.json')

fixed_252_regime_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_seed_raw.csv')
fixed_252_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_window_summary.csv')
fixed_252_regime_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_bucket_summary.csv')

fixed_252_year_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_seed_raw.csv')
fixed_252_year_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_window_summary.csv')
fixed_252_year_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_bucket_summary.csv')

matched_regime_raw_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_seed_raw.csv')
matched_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_window_summary.csv')
matched_regime_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_bucket_summary.csv')

benchmark_raw_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv')
benchmark_summary_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmark_summary.csv')

manifest

fixed_252_regime_cvar_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_cvar_raw.csv')
fixed_252_regime_cvar_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_cvar_window_summary.csv')
fixed_252_regime_cvar_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_regime']['tables'] / 'fixed_252_regime_cvar_bucket_summary.csv')
fixed_252_year_cvar_raw_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_cvar_raw.csv')
fixed_252_year_cvar_window_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_cvar_window_summary.csv')
fixed_252_year_cvar_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['fixed_252_year']['tables'] / 'fixed_252_year_cvar_bucket_summary.csv')
matched_regime_cvar_raw_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_cvar_raw.csv')
matched_regime_cvar_window_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_cvar_window_summary.csv')
matched_regime_cvar_bucket_summary_df = load_csv_or_empty(SECTION_DIRS['matched_regime']['tables'] / 'matched_regime_cvar_bucket_summary.csv')
cvar_stage_overview_df = load_csv_or_empty(SECTION_DIRS['cvar']['tables'] / 'stochastic_cvar_stage_overview.csv')
benchmark_cvar_summary_df = load_csv_or_empty(SECTION_DIRS['cvar']['tables'] / 'benchmark_cvar_summary.csv')


## 15) Contained-Window Regime Appendix
These windows are fully contained within each regime bucket. They are stricter than the anchored windows and are intended as an attribution-purity appendix.


In [ ]:
contained_regime_plan_df = build_contained_regime_windows(
    test_dates,
    horizon_days=FIXED_252_HORIZON_DAYS,
    windows_per_bucket=REGIME_WINDOWS_PER_BUCKET,
)
contained_regime_plan_df['years'] = 1
contained_regime_plan_df['stage'] = 'contained_regime'

display(contained_regime_plan_df)
save_df(contained_regime_plan_df, SECTION_DIRS['contained_regime']['plans'] / 'contained_regime_plan.csv')


In [ ]:
contained_regime_run_df = run_plan_rows(
    contained_regime_plan_df,
    stage_key='contained_regime',
    stage_dirs=SECTION_DIRS['contained_regime'],
    num_eval_runs=FIXED_252_STOCH_RUNS,
    limit_rows=RUN_LIMIT_FIXED_252_REGIME,
)
contained_regime_raw_df = load_csv_or_empty(SECTION_DIRS['contained_regime']['tables'] / 'contained_regime_seed_raw.csv')
contained_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['contained_regime']['tables'] / 'contained_regime_window_summary.csv')

display(contained_regime_run_df)
display(contained_regime_window_summary_df.tail(10))


In [ ]:
contained_regime_raw_df = load_csv_or_empty(SECTION_DIRS['contained_regime']['tables'] / 'contained_regime_seed_raw.csv')
contained_regime_window_summary_df = load_csv_or_empty(SECTION_DIRS['contained_regime']['tables'] / 'contained_regime_window_summary.csv')
contained_regime_bucket_summary_df = summarize_bucket_degradation(contained_regime_window_summary_df)
contained_regime_cvar_raw_df, contained_regime_cvar_window_summary_df, contained_regime_cvar_bucket_summary_df = refresh_stage_cvar_tables('contained_regime', SECTION_DIRS['contained_regime'])

save_df(contained_regime_bucket_summary_df, SECTION_DIRS['contained_regime']['tables'] / 'contained_regime_bucket_summary.csv')

display(contained_regime_bucket_summary_df)
display(contained_regime_cvar_bucket_summary_df)
